# 04 — Gold Business Data & AI/BI Dashboards: Real Estate (Property Listings, Market Valuation & AVM Pricing)

**Purpose:** Aggregate Silver tables into executive-ready Gold KPI tables (`real_estate_prod.gold.gold_city_market_kpis` and `real_estate_prod.gold.gold_neighborhood_price_trends`) and verify SQL queries for Databricks AI/BI Dashboards (Lakeview).

In [ ]:
# Databricks Widget Parameters (overridden by Lakeflow Jobs / Databricks Asset Bundles)
dbutils.widgets.text("catalog_name", "real_estate_prod", "Unity Catalog Name")
CATALOG = dbutils.widgets.get("catalog_name")
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_landing"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/bronze/checkpoints"
print(f"Active Catalog: {CATALOG} | Landing Volume: {VOLUME_PATH}")

## Step 1: Build Gold Business KPI Tables

In [ ]:
from pyspark.sql import functions as F

props = spark.table(f"{CATALOG}.silver.silver_properties_clean")
sales = spark.table(f"{CATALOG}.silver.silver_sales_enriched")

gold_city = (
    props.groupBy("city", "property_type")
    .agg(
        F.count("property_id").alias("active_listings"),
        F.round(F.avg("list_price_usd"), 0).alias("avg_list_price_usd"),
        F.round(F.avg("price_per_sqft"), 2).alias("avg_price_per_sqft"),
        F.round(F.avg("days_on_market"), 1).alias("avg_days_on_market"),
    )
)
gold_city.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.gold_city_market_kpis")

gold_trends = (
    sales.withColumn("close_month", F.date_format("close_date", "yyyy-MM"))
    .groupBy("city", "close_month")
    .agg(
        F.count("sale_id").alias("closed_sales_count"),
        F.round(F.sum("closed_price_usd"), 0).alias("total_sales_volume_usd"),
        F.round(F.avg("sale_to_list_ratio"), 3).alias("avg_sale_to_list_ratio"),
    )
)
gold_trends.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.gold_neighborhood_price_trends")

display(gold_city.orderBy(F.col("avg_price_per_sqft").desc()))
display(gold_trends.orderBy("close_month", "city").limit(12))

## Step 2: Execute & Validate AI/BI Dashboard Queries

In [ ]:
# Dashboard Query 1: Average Price per SqFt by City & Type
q_1 = """SELECT city, property_type, active_listings, avg_list_price_usd, avg_price_per_sqft, avg_days_on_market FROM {CATALOG}.gold.gold_city_market_kpis ORDER BY avg_price_per_sqft DESC;""".format(CATALOG=CATALOG)
display(spark.sql(q_1))

# Dashboard Query 2: Monthly Closed Sales Volume by City
q_2 = """SELECT close_month, city, closed_sales_count, total_sales_volume_usd, avg_sale_to_list_ratio FROM {CATALOG}.gold.gold_neighborhood_price_trends ORDER BY close_month, city;""".format(CATALOG=CATALOG)
display(spark.sql(q_2))

## Step 3: Optimize Gold Tables for Low-Latency SQL Warehouse Dashboards

In [ ]:
for tbl in ["gold_city_market_kpis", "gold_neighborhood_price_trends"]:
    spark.sql(f"OPTIMIZE {CATALOG}.gold.{tbl}")
print("Gold tables optimized for AI/BI Dashboards.")